# ENM prototyping

Can I recreate the elastic network model for tetragonal lysozyme in python?

Pre-exercises:

1. Load `test_data/lys_1_enm_edges.csv` as a pandas dataframe
2. Load `test_data/lys_1_refmac.pdb` as a gemmi structure
3. Look up the atoms in the structure by their location (including symmetry mates)
4. Cross-check with the contact list generated by Gemmi

### Ex. 1

In [1]:
import pandas as pd

df = pd.read_csv('test_data/lys_1_enm_edges.csv')

def group_cols(df, cols, name):
    df[name] = df[cols].apply(tuple, axis=1)
    df.drop(columns=cols,inplace=True)
    return df

group_cols(df,['c2_1', 'c2_2', 'c2_3'],'c2')
group_cols(df,['r1_1', 'r1_2', 'r1_3'],'r1')
group_cols(df,['r2_1', 'r2_2', 'r2_3'],'r2')

df.head()

,o1,o2,interface,a1,a2,g1,g2,c2,r1,r2
0,1,3,1,72,63,112,102,"(1, 0, 0)","(6.159, 33.583, 23.441)","(8.26510000000001, 35.0001, 21.6509)"
1,1,3,1,73,71,113,106,"(1, 0, 0)","(4.026, 34.598, 17.097)","(5.9891, 37.4471, 15.7779)"
2,1,3,1,74,71,113,106,"(1, 0, 0)","(5.494, 34.189, 17.224)","(5.9891, 37.4471, 15.7779)"
3,1,3,1,75,63,113,102,"(1, 0, 0)","(6.064, 34.598, 18.558)","(8.26510000000001, 35.0001, 21.6509)"
4,1,3,1,75,64,113,103,"(1, 0, 0)","(6.064, 34.598, 18.558)","(8.8721, 37.0161, 19.7799)"


### Ex. 2

In [3]:
import gemmi

st = gemmi.read_structure('test_data/lys_1_refmac.pdb')
st.setup_entities() # supposed to be good practice
st[0].remove_ligands_and_waters()

### Ex. 3

In [4]:
ns = gemmi.NeighborSearch(st[0], st.cell, 5).populate(include_h=True) # not sure if I used hydrogens or not...

def lookup_coordinates(row):
    p1 = gemmi.Position(*row['r1'])
    p2 = gemmi.Position(*row['r2'])
    m1 = ns.find_nearest_atom(p1)
    m2 = ns.find_nearest_atom(p2)
    cra1 = m1.to_cra(st[0])
    cra2 = m2.to_cra(st[0])
    im1 = st.cell.find_nearest_pbc_image(p1, cra1.atom.pos, m1.image_idx) # point, cra.atom.pos, mark.image_idx
    im2 = st.cell.find_nearest_pbc_image(p2, cra2.atom.pos, m2.image_idx)
    if im1.dist() > .01:
        raise ValueError('nearest pbc image of atom 1 in structure is too far away')
    if im2.dist() > .01:
        raise ValueError('nearest pbc image of atom 2 in structure is too far away')
    row['cra1'] = str(cra1)
    row['cra2'] = str(cra2)
    row['sym_idx1'] = im1.sym_idx
    row['sym_idx2'] = im2.sym_idx
    row['pbc_shift1'] = im1.pbc_shift
    row['pbc_shift2'] = im2.pbc_shift
    return row

df2 = df.apply(lookup_coordinates,axis=1)
df2.head()

,o1,o2,interface,a1,a2,g1,g2,c2,r1,r2,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
0,1,3,1,72,63,112,102,"(1, 0, 0)","(6.159, 33.583, 23.441)","(8.26510000000001, 35.0001, 21.6509)",A/ARG 112/NH2,A/GLY 102/O,0,1,"(0, 0, 0)","(0, 0, -1)"
1,1,3,1,73,71,113,106,"(1, 0, 0)","(4.026, 34.598, 17.097)","(5.9891, 37.4471, 15.7779)",A/ASN 113/CA,A/ASN 106/ND2,0,1,"(0, 0, 0)","(0, 0, -1)"
2,1,3,1,74,71,113,106,"(1, 0, 0)","(5.494, 34.189, 17.224)","(5.9891, 37.4471, 15.7779)",A/ASN 113/CB,A/ASN 106/ND2,0,1,"(0, 0, 0)","(0, 0, -1)"
3,1,3,1,75,63,113,102,"(1, 0, 0)","(6.064, 34.598, 18.558)","(8.26510000000001, 35.0001, 21.6509)",A/ASN 113/CG,A/GLY 102/O,0,1,"(0, 0, 0)","(0, 0, -1)"
4,1,3,1,75,64,113,103,"(1, 0, 0)","(6.064, 34.598, 18.558)","(8.8721, 37.0161, 19.7799)",A/ASN 113/CG,A/ASN 103/CA,0,1,"(0, 0, 0)","(0, 0, -1)"


In [5]:
# lets see how op1 and sym_idx1 match up:
df2.groupby(['o1','sym_idx1']).size().reset_index().rename(columns={0:'count'})

,o1,sym_idx1,count
0,1,0,202
1,2,2,202
2,3,1,163
3,4,6,137
4,5,3,138
5,6,4,137
6,7,5,163
7,8,7,163


In [6]:
# lets see how op2 and sym_idx2 match up:
df2.groupby(['o2','sym_idx2']).size().reset_index().rename(columns={0:'count'})

,o2,sym_idx2,count
0,1,0,163
1,2,2,163
2,3,1,137
3,4,6,138
4,5,3,137
5,6,4,163
6,7,5,202
7,8,7,202


In [7]:
# lets see how interface is encoded
df2[df2['o1']==1].groupby(['o2','interface']).size().reset_index().rename(columns={0:'count'})

,o2,interface,count
0,3,1,39
1,3,2,15
2,5,-2,15
3,5,-1,39
4,7,3,25
5,8,4,65
6,8,5,4


### Ex 4. 

Cross-check with the contact list generated by Gemmi

1. perform a contact search
2. Create a table of CRA strings, pdbshifts, etc.
3. Create the corresponding table from the MATLAB output
4. Check if any rows are different / missing (merge?)

In [9]:
cs = gemmi.ContactSearch(4.0)
cs.ignore = gemmi.ContactSearch.Ignore.SameAsu 
ns = gemmi.NeighborSearch(st[0], st.cell, 5).populate(include_h=False)
results = cs.find_contacts(ns)
len(results)

102

In [14]:
# make the table

    # row['cra1'] = str(cra1)
    # row['cra2'] = str(cra2)
    # row['sym_idx1'] = im1.sym_idx
    # row['sym_idx2'] = im2.sym_idx
    # row['pbc_shift1'] = im1.pbc_shift
    # row['pbc_shift2'] = im2.pbc_shift

def results2dict(contacts):
    d = {
        'cra1':[str(res.partner1) for res in contacts],
        'cra2':[str(res.partner2) for res in contacts],
        'sym_idx1':[0 for res in contacts],
        'sym_idx2':[res.image_idx for res in contacts],
        'pbc_shift1':[(0,0,0) for res in contacts],
        'pbc_shift2':[st.cell.find_nearest_pbc_image(
            res.partner1.atom.pos, 
            res.partner2.atom.pos, 
            res.image_idx).pbc_shift for res in contacts],
    }
    return d

df3 = pd.DataFrame.from_dict(results2dict(results))
df3

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, 0)"
1,A/LYS 13/CE,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
3,A/LYS 13/NZ,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, 0)"
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, 0)"
...,...,...,...,...,...,...
97,A/ASN 106/ND2,A/ASN 113/CG,0,3,"(0, 0, 0)","(-1, 0, 0)"
98,A/ASN 106/ND2,A/ASN 113/OD1,0,3,"(0, 0, 0)","(-1, 0, 0)"
99,A/ASN 113/O,A/LYS 116/CE,0,1,"(0, 0, 0)","(0, 0, -1)"
100,A/ASN 113/O,A/LYS 116/NZ,0,1,"(0, 0, 0)","(0, 0, -1)"


In [11]:
# how many CRAs are there for the ASU in the matlab version?

pd.concat((df2[(df2['o1']==1)]['cra1'], df2[(df2['o2']==1)]['cra2'])).value_counts()

A/GLN 41/OE1     20
A/ASN 113/ND2    14
A/THR 47/O       12
A/ASN 113/CG     10
A/ASN 113/OD1    10
                 ..
A/GLY 104/CA      1
A/ASN 106/CG      1
A/ASN 106/OD1     1
A/LYS 116/CE      1
A/LYS 116/NZ      1
Name: count, Length: 99, dtype: int64

In [12]:
# the same CRAs evidently appear uniquely in the "o1" group

df2[(df2['o1']==1)].groupby('cra1').size()

cra1
A/ALA 10/CB       1
A/ALA 42/CA       1
A/ARG 112/NH2     1
A/ARG 114/CD      3
A/ARG 114/CG      1
                 ..
A/THR 47/OG1.A    2
A/TYR 23/CD1      3
A/TYR 23/CE1      4
A/TYR 23/CZ       1
A/TYR 23/OH       2
Length: 99, dtype: int64

In [13]:
# I found fewer in gemmi. Why?

df3.groupby('cra1').size()

cra1
A/ALA 10/CB        1
A/ALA 42/CA        1
A/ARG 14/O         2
A/ASN 103/C        3
A/ASN 103/CA       3
A/ASN 103/O        3
A/ASN 106/CG       1
A/ASN 106/ND2      6
A/ASN 106/OD1      1
A/ASN 113/O        2
A/ASN 27/ND2       1
A/ASN 39/CG        1
A/ASN 39/ND2       5
A/ASN 39/OD1       1
A/ASN 44/ND2       1
A/ASP 48/CA        1
A/ASP 48/CB        1
A/GLN 41/CB        1
A/GLN 41/CD        2
A/GLN 41/NE2       4
A/GLN 41/OE1      10
A/GLY 102/C        1
A/GLY 102/O        4
A/GLY 104/CA       1
A/GLY 104/N        1
A/GLY 16/CA        3
A/GLY 16/N         1
A/GLY 22/C         1
A/GLY 22/O         1
A/LEU 129/O        1
A/LEU 84/CD1       1
A/LYS 13/CE        2
A/LYS 13/NZ        2
A/LYS 13/O         1
A/PRO 70/CB        2
A/THR 43/C         1
A/THR 43/N         1
A/THR 43/O         3
A/THR 43/OG1       1
A/THR 47/CB.A      1
A/THR 47/CB.B      1
A/THR 47/CG2.B     3
A/THR 47/O         6
A/THR 47/OG1.A     2
A/TYR 23/CD1       3
A/TYR 23/CE1       4
A/TYR 23/CZ        1
A/TYR 23

In [16]:
# let's check the length of the bonds
import numpy as np

d = []
for j, row in enumerate(df2.itertuples()):
    d.append(np.linalg.norm(np.array(row.r1) - np.array(row.r2)))
np.array(d).max()

np.float64(3.9928114355676785)

In [17]:
# values of CRA1 from Gemmi that are not in CRA1 list from MATLAB
df3[~df3['cra1'].isin(df2['cra1'])]

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2


In [18]:
# values of CRA1 from MATLAB that are not in CRA1 list from Gemmi
tmp = df2[~df2['cra1'].isin(df3['cra1'])]
tmp[tmp['o1']==1].groupby('cra1').size()

cra1
A/ARG 112/NH2    1
A/ARG 114/CD     3
A/ARG 114/CG     1
A/ARG 114/CZ     2
A/ARG 114/NE     1
A/ARG 114/NH1    4
A/ARG 114/NH2    2
A/ARG 128/CB     1
A/ARG 128/CD     2
A/ARG 128/CG     1
A/ARG 128/CZ     3
A/ARG 128/NE     5
A/ARG 128/NH1    2
A/ARG 128/NH2    1
A/ARG 14/CD      1
A/ARG 45/NH1     1
A/ARG 68/CB      1
A/ARG 68/CZ      1
A/ARG 68/NH1     4
A/ARG 68/NH2     2
A/ASN 113/C      1
A/ASN 113/CA     1
A/ASN 113/CB     1
A/ASN 113/CG     5
A/ASN 113/ND2    7
A/ASN 113/OD1    5
A/ASN 65/C       1
A/ASN 65/O       3
A/ASP 66/C       1
A/ASP 66/CA      2
A/ASP 66/O       4
A/CYS 127/O      2
A/CYS 80/C       1
A/CYS 80/CA      1
A/CYS 80/CB      1
A/CYS 80/N       1
A/GLY 126/C      1
A/GLY 126/O      4
A/GLY 67/C       1
A/GLY 67/CA      1
A/GLY 67/O       1
A/GLY 71/CA      1
A/GLY 71/N       1
A/LEU 129/C      2
A/LYS 116/CE     1
A/LYS 116/NZ     1
A/PRO 79/C       1
A/SER 81/CA      1
A/SER 81/CB      1
A/SER 81/N       1
A/SER 81/OG      1
dtype: int64

In [19]:
# lets see if we can figure out why. Choose one to start with:

df2[(df2['cra1'] == 'A/ARG 114/CD') & (df2['o1']==1)]

,o1,o2,interface,a1,a2,g1,g2,c2,r1,r2,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
27,1,3,1,81,12,114,23,"(1, 0, 0)","(5.717, 31.314, 12.966)","(9.0121, 32.7421, 12.2579)",A/ARG 114/CD,A/TYR 23/CE1,0,1,"(0, 0, 0)","(0, 0, -1)"
28,1,3,1,81,13,114,23,"(1, 0, 0)","(5.717, 31.314, 12.966)","(9.24210000000001, 32.5761, 13.6119)",A/ARG 114/CD,A/TYR 23/CZ,0,1,"(0, 0, 0)","(0, 0, -1)"
29,1,3,1,81,14,114,23,"(1, 0, 0)","(5.717, 31.314, 12.966)","(8.4421, 33.2571, 14.5069)",A/ARG 114/CD,A/TYR 23/OH,0,1,"(0, 0, 0)","(0, 0, -1)"


In [20]:
df3[(df3['cra1'] == 'A/ARG 114/CD') | (df3['cra2'] == 'A/ARG 114/CD')]

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
20,A/TYR 23/CE1,A/ARG 114/CD,0,3,"(0, 0, 0)","(-1, 0, 0)"
21,A/TYR 23/CZ,A/ARG 114/CD,0,3,"(0, 0, 0)","(-1, 0, 0)"
23,A/TYR 23/OH,A/ARG 114/CD,0,3,"(0, 0, 0)","(-1, 0, 0)"


Oh, weird! Actually these bonds are present. Contact search is avoiding duplicates, which is a good thing, but it makes comparison with MATLAB more difficult



In [21]:
# values of CRA1 from MATLAB that are not in CRA1 or CRA2 lists from Gemmi
tmp = df2[ ~((df2['cra1'].isin(df3['cra1'])) | (df2['cra1'].isin(df3['cra2']))) ]
tmp[tmp['o1']==1].groupby('cra1').size()

Series([], dtype: int64)

In [22]:
# is this correct? let's check another way
df2_cras = pd.concat((df2['cra1'],df2['cra2']))
df3_cras = pd.concat((df3['cra1'],df3['cra2']))
len(df2_cras.unique()), len(df3_cras.unique()), df2_cras.isin(df3_cras).all(), df3_cras.isin(df2_cras).all()

(99, 99, np.True_, np.True_)

In [23]:
# ah, interesting! now lets compare atom pairs, regardless of order

tmp1 = df2[df2['o1']==1][['cra1','cra2']]
tmp2 = df2[df2['o2']==1][['cra1','cra2']].rename(columns={'cra1': 'cra2', 'cra2': 'cra1'})
tmp = pd.concat((tmp1,tmp2))
tmp['sorted_pair'] = tmp.apply(lambda row: tuple(sorted([row['cra1'], row['cra2']])), axis=1)
tmp.groupby('sorted_pair').size()

sorted_pair
(A/ALA 10/CB, A/ARG 14/CD)      4
(A/ALA 42/CA, A/ARG 68/NH1)     4
(A/ARG 112/NH2, A/GLY 102/O)    3
(A/ARG 114/CD, A/TYR 23/CE1)    3
(A/ARG 114/CD, A/TYR 23/CZ)     3
                               ..
(A/LEU 129/C, A/LYS 13/NZ)      4
(A/LEU 129/O, A/LEU 129/O)      2
(A/LEU 129/O, A/LYS 13/CE)      4
(A/LEU 129/O, A/LYS 13/NZ)      4
(A/LEU 84/CD1, A/LEU 84/CD1)    2
Length: 102, dtype: int64

In [24]:
tmp3 = df3[['cra1','cra2']].copy()
tmp3['sorted_pair'] = tmp3.apply(lambda row: tuple(sorted([row['cra1'], row['cra2']])), axis=1)
tmp3.groupby('sorted_pair').size()

sorted_pair
(A/ALA 10/CB, A/ARG 14/CD)      1
(A/ALA 42/CA, A/ARG 68/NH1)     1
(A/ARG 112/NH2, A/GLY 102/O)    1
(A/ARG 114/CD, A/TYR 23/CE1)    1
(A/ARG 114/CD, A/TYR 23/CZ)     1
                               ..
(A/LEU 129/C, A/LYS 13/NZ)      1
(A/LEU 129/O, A/LEU 129/O)      1
(A/LEU 129/O, A/LYS 13/CE)      1
(A/LEU 129/O, A/LYS 13/NZ)      1
(A/LEU 84/CD1, A/LEU 84/CD1)    1
Length: 102, dtype: int64

### CONCLUSIONS

1. MATLAB and Gemmi are finding the same contacts
2. The labeling of unit cell operators is different. This was expected, because in MATLAB I mapped everything to a particular box.
3. The next step is to figure out the symmetry expansion algorithm